In [2]:
import pandas as pd

results = pd.read_csv("../outputs/triage_results.csv")

print(results.shape)
print(results["processing_status"].value_counts())
print(results["topic"].value_counts())
print(results["urgency"].value_counts())
print(results["needs_more_information"].value_counts())
print(results["processing_time_seconds"].describe())
print(results["next_action"].value_counts())

(200, 12)
processing_status
success    200
Name: count, dtype: int64
topic
Technical / Online Access    109
Other                         65
Policy / Contract             17
Billing / Payment              8
Claims / Damage                1
Name: count, dtype: int64
urgency
Medium    115
Low        47
High       38
Name: count, dtype: int64
needs_more_information
False    190
True      10
Name: count, dtype: int64
count    200.000000
mean      22.277150
std        5.486147
min       13.660000
25%       19.040000
50%       20.365000
75%       23.967500
max       45.000000
Name: processing_time_seconds, dtype: float64
next_action
Forward to technical support         70
Route to general customer service    60
Escalate to human supervisor         36
Route to policy service              16
Ask customer for more information    10
Forward to billing team               8
Name: count, dtype: int64


In [3]:
total_tickets = len(results)
success_count = results["processing_status"].eq("success").sum()
fallback_count = results["processing_status"].eq("fallback").sum()

success_rate = success_count / total_tickets
fallback_rate = fallback_count / total_tickets

print(f"Total tickets: {total_tickets}")
print(f"Success rate: {success_rate:.2%}")
print(f"Fallback rate: {fallback_rate:.2%}")
print(
    f"Average processing time: "
    f"{results['processing_time_seconds'].mean():.2f} seconds"
)
print(
    f"Median processing time: "
    f"{results['processing_time_seconds'].median():.2f} seconds"
)
print(
    f"Tickets per minute: "
    f"{60 / results['processing_time_seconds'].mean():.2f}"
)

Total tickets: 200
Success rate: 100.00%
Fallback rate: 0.00%
Average processing time: 22.28 seconds
Median processing time: 20.37 seconds
Tickets per minute: 2.69


In [4]:
required_columns = [
    "ticket_id",
    "topic",
    "urgency",
    "needs_more_information",
    "next_action",
    "notes",
    "processing_status",
]

missing_required_values = (
    results[required_columns]
    .isna()
    .sum()
)

print(missing_required_values)

ticket_id                 0
topic                     0
urgency                   0
needs_more_information    0
next_action               0
notes                     0
processing_status         0
dtype: int64


In [5]:
empty_string_counts = {}

for column in required_columns:
    if results[column].dtype == "object":
        empty_string_counts[column] = (
            results[column]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )

pd.Series(empty_string_counts)

Series([], dtype: object)

In [6]:
complete_rows = results[required_columns].notna().all(axis=1)

schema_completeness = complete_rows.mean()

print(f"Schema completeness: {schema_completeness:.2%}")

Schema completeness: 100.00%


In [7]:
allowed_topics = {
    "Policy / Contract",
    "Claims / Damage",
    "Billing / Payment",
    "Technical / Online Access",
    "Other",
}

allowed_urgencies = {
    "Low",
    "Medium",
    "High",
}

invalid_topics = results[
    ~results["topic"].isin(allowed_topics)
]

invalid_urgencies = results[
    ~results["urgency"].isin(allowed_urgencies)
]

print("Invalid topics:", len(invalid_topics))
print("Invalid urgencies:", len(invalid_urgencies))

Invalid topics: 0
Invalid urgencies: 0


In [12]:
invalid_high_routing = results[
    results["urgency"].eq("High")
    & ~results["next_action"].eq(
        "Escalate to human supervisor"
    )
]

print(
    "High urgency routing violations:",
    len(invalid_high_routing),
)
invalid_high_routing.head()

High urgency routing violations: 2


,ticket_id,subject,body,topic,urgency,needs_more_information,missing_information,clarification_question,next_action,notes,processing_status,processing_time_seconds
62,6707,Request for Investment Support,"Dear Customer Support, our investment optimiza...",Technical / Online Access,High,True,"[""A clearer description of the customer's issue""]",Could you provide more details about the speci...,Ask customer for more information,The customer is requesting assistance with a t...,success,27.2
154,23115,NaN,Attention needed for data breach found in the ...,Technical / Online Access,High,True,"[""A clearer description of the customer's issue""]",Could you provide more details about the natur...,Ask customer for more information,Data breach in hospital's system constitutes a...,success,23.7


In [13]:
invalid_missing_info_routing = results[
    results["needs_more_information"].eq(True)
    & ~results["next_action"].eq(
        "Ask customer for more information"
    )
]

print(
    "Missing-information routing violations:",
    len(invalid_missing_info_routing),
)

Missing-information routing violations: 0


In [14]:
missing_clarification_questions = results[
    results["needs_more_information"].eq(True)
    & (
        results["clarification_question"].isna()
        | results["clarification_question"]
        .fillna("")
        .str.strip()
        .eq("")
    )
]

print(
    "Missing clarification questions:",
    len(missing_clarification_questions),
)

Missing clarification questions: 0


In [15]:
unexpected_clarification_questions = results[
    results["needs_more_information"].eq(False)
    & results["clarification_question"].notna()
    & results["clarification_question"]
    .fillna("")
    .str.strip()
    .ne("")
]

print(
    "Unexpected clarification questions:",
    len(unexpected_clarification_questions),
)

Unexpected clarification questions: 0


In [16]:
invalid_missing_information = results[
    results["needs_more_information"].eq(True)
    & (
        results["missing_information"].isna()
        | results["missing_information"]
        .fillna("")
        .astype(str)
        .str.strip()
        .isin(["", "[]"])
    )
]

print(
    "Missing missing-information details:",
    len(invalid_missing_information),
)

Missing missing-information details: 0


In [18]:
results["is_consistent"] = True

results.loc[
    results.index.isin(invalid_high_routing.index),
    "is_consistent",
] = False

results.loc[
    results.index.isin(invalid_missing_info_routing.index),
    "is_consistent",
] = False

results.loc[
    results.index.isin(missing_clarification_questions.index),
    "is_consistent",
] = False

results.loc[
    results.index.isin(unexpected_clarification_questions.index),
    "is_consistent",
] = False

results.loc[
    results.index.isin(invalid_missing_information.index),
    "is_consistent",
] = False

consistency_rate = results["is_consistent"].mean()

print(f"Workflow consistency rate: {consistency_rate:.2%}")

Workflow consistency rate: 99.00%


In [19]:
pd.crosstab(
    results["topic"],
    results["urgency"],
)

urgency,High,Low,Medium
topic,,,
Billing / Payment,0,2,6
Claims / Damage,1,0,0
Other,2,37,26
Policy / Contract,1,1,15
Technical / Online Access,34,7,68


In [20]:
pd.crosstab(
    results["topic"],
    results["next_action"],
)

next_action,Ask customer for more information,Escalate to human supervisor,Forward to billing team,Forward to technical support,Route to general customer service,Route to policy service
topic,,,,,,
Billing / Payment,0,0,8,0,0,0
Claims / Damage,0,1,0,0,0,0
Other,3,2,0,0,60,0
Policy / Contract,0,1,0,0,0,16
Technical / Online Access,7,32,0,70,0,0


In [21]:
pd.crosstab(
    results["needs_more_information"],
    results["next_action"],
)

next_action,Ask customer for more information,Escalate to human supervisor,Forward to billing team,Forward to technical support,Route to general customer service,Route to policy service
needs_more_information,,,,,,
False,0,36,8,70,60,16
True,10,0,0,0,0,0
